In [ ]:
import os, json, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from dotenv import load_dotenv
from datasets import load_dataset
from openai import OpenAI
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score

load_dotenv(override=True)
assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY in env/.env"

client = OpenAI()
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
ds = load_dataset("ed-donner/items_lite")

train_df = pd.DataFrame(ds["train"])
test_df  = pd.DataFrame(ds["test"])

# Keep only what we need
train_df = train_df[["summary", "price"]].dropna()
test_df  = test_df[["summary", "price"]].dropna()

# Basic filtering
train_df = train_df[train_df["summary"].str.len() > 30]
test_df  = test_df[test_df["summary"].str.len() > 30]
train_df = train_df[train_df["price"] > 0]
test_df  = test_df[test_df["price"] > 0]

MAX_CHARS = 800  # try 600–1000 for speed/cost tradeoff
train_df["summary"] = train_df["summary"].str.slice(0, MAX_CHARS)
test_df["summary"]  = test_df["summary"].str.slice(0, MAX_CHARS)

print("✅ Loaded:", len(train_df), "train |", len(test_df), "test")
train_df.head(2)

In [ ]:
baseline_mean = train_df["price"].mean()
baseline_pred = np.full(len(test_df), baseline_mean)
baseline_mae  = mean_absolute_error(test_df["price"].values, baseline_pred)

print(f"Baseline mean price: ${baseline_mean:,.2f}")
print(f"Baseline MAE:        ${baseline_mae:,.2f}")

In [ ]:
# Price distribution
plt.figure()
plt.hist(train_df["price"], bins=50)
plt.title("Train Price Distribution")
plt.xlabel("Price ($)")
plt.ylabel("Count")
plt.show()

# Price tiers
bins = [0, 20, 50, 100, 1000]
labels = ["Budget (<$20)", "Mid ($20-$50)", "Premium ($50-$100)", "Luxury (>$100)"]
tiers = pd.cut(train_df["price"], bins=bins, labels=labels)
tier_counts = tiers.value_counts()

plt.figure()
plt.pie(tier_counts.values, labels=tier_counts.index, autopct="%1.1f%%")
plt.title("Train Set Composition by Price Tier")
plt.show()

In [ ]:
N_BINS = 20  # P00..P19

kbd = KBinsDiscretizer(n_bins=N_BINS, encode="ordinal", strategy="quantile")

train_bins = kbd.fit_transform(train_df[["price"]]).astype(int).ravel()
test_bins  = kbd.transform(test_df[["price"]]).astype(int).ravel()

def bin_label(b: int) -> str:
    return f"P{b:02d}"

train_df["label"] = [bin_label(b) for b in train_bins]
test_df["label"]  = [bin_label(b) for b in test_bins]

# Map label -> median price in that bin (from training set)
train_df["bin_id"] = train_bins
bin_medians = train_df.groupby("bin_id")["price"].median().to_dict()
label_to_price = {bin_label(b): float(m) for b, m in bin_medians.items()}

print("Example labels:", list(label_to_price.items())[:5])

In [ ]:
SYSTEM_PROMPT = (
    "You are a pricing model. Given a product description, predict its price bucket.\n"
    "Output ONLY one label in this format: P00, P01, ..., P19.\n"
    "No extra text."
)

def to_jsonl_row(summary: str, label: str):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": summary},
            {"role": "assistant", "content": label},
        ]
    }

def write_jsonl(df: pd.DataFrame, path: str, limit: int):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for _, r in df.head(limit).iterrows():
            f.write(json.dumps(to_jsonl_row(r["summary"], r["label"]), ensure_ascii=False) + "\n")

TRAIN_LIMIT = 100
VAL_LIMIT   = 50

# simple split from train_df
val_split = train_df.sample(n=min(VAL_LIMIT, len(train_df)), random_state=42)
train_split = train_df.drop(val_split.index)

train_path = "ft_data/train.jsonl"
val_path   = "ft_data/val.jsonl"

write_jsonl(train_split, train_path, TRAIN_LIMIT)
write_jsonl(val_split,   val_path,   VAL_LIMIT)

print("✅ Wrote:", train_path, val_path)

In [ ]:
from pathlib import Path

train_file = client.files.create(file=Path(train_path), purpose="fine-tune")
val_file   = client.files.create(file=Path(val_path),   purpose="fine-tune")

print("train_file_id:", train_file.id)
print("val_file_id:",   val_file.id)

In [ ]:
BASE_MODEL = "gpt-4o-mini-2024-07-18"

job = client.fine_tuning.jobs.create(
    model=BASE_MODEL,
    training_file=train_file.id,
    validation_file=val_file.id,
)

print("job_id:", job.id)
print("status:", job.status)
JOB_ID = job.id

In [ ]:
j = client.fine_tuning.jobs.retrieve(JOB_ID)
print("status:", j.status)
print("error:", getattr(j, "error", None))

# Also print events (usually shows the real reason)
events = client.fine_tuning.jobs.list_events(JOB_ID, limit=20)
for e in events.data[::-1]:
    print(f"{e.created_at} | {e.level} | {e.message}")

In [ ]:
while True:
    j = client.fine_tuning.jobs.retrieve(JOB_ID)
    print("status:", j.status)
    if j.status in ("succeeded", "failed", "cancelled"):
        print("done:", j.status)
        print("fine_tuned_model:", j.fine_tuned_model)
        break
    time.sleep(20)

FINE_TUNED_MODEL = j.fine_tuned_model
assert FINE_TUNED_MODEL, "Fine-tuned model not available (job may have failed)."

In [ ]:
LABEL_RE = re.compile(r"P\d{2}")

def predict_label(summary: str) -> str:
    resp = client.chat.completions.create(
        model=FINE_TUNED_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": summary},
        ],
        temperature=0,
        max_tokens=10,
    )
    out = resp.choices[0].message.content.strip()
    m = LABEL_RE.search(out)
    return m.group(0) if m else "P00"

TEST_N = min(1000, len(test_df))  # keep it cheap
pred_labels = [predict_label(s) for s in test_df["summary"].head(TEST_N)]
true_labels = test_df["label"].head(TEST_N).tolist()

pred_prices = [label_to_price.get(lb, label_to_price["P00"]) for lb in pred_labels]
true_prices = test_df["price"].head(TEST_N).tolist()

print("Bucket accuracy:", accuracy_score(true_labels, pred_labels))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

y_true = np.array(true_prices, dtype=float)
y_pred = np.array(pred_prices, dtype=float)

mae = mean_absolute_error(y_true, y_pred)

mse = mean_squared_error(y_true, y_pred)   # no squared=...
rmse = float(np.sqrt(mse))

r2 = r2_score(y_true, y_pred)

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R²:   {r2:.4f}")
print(f"(Baseline MAE was ~${baseline_mae:,.2f})")

# 1) Predicted vs Actual
plt.figure()
plt.scatter(y_true, y_pred, s=10, alpha=0.35)
plt.xlabel("Actual Price ($)")
plt.ylabel("Predicted Price ($)")
plt.title("Predicted vs Actual (Bucket Median)")
plt.show()

# 2) Residual distribution
resid = y_pred - y_true
plt.figure()
plt.hist(resid, bins=50)
plt.xlabel("Residual ($) = Predicted - Actual")
plt.ylabel("Count")
plt.title("Residual Distribution")
plt.show()

In [ ]:
def answer_price(desc: str):
    lb = predict_label(desc)
    est = label_to_price.get(lb, label_to_price["P00"])
    return f"Predicted bucket: {lb}\nEstimated price: ${est:,.2f}"

with gr.Blocks() as demo:
    gr.Markdown("# 🧾 Price Bucket Predictor (Fine-tuned OpenAI)")
    inp = gr.Textbox(label="Product description", lines=6)
    out = gr.Textbox(label="Prediction")
    btn = gr.Button("Predict")
    btn.click(answer_price, inputs=[inp], outputs=[out])

demo.launch(inbrowser=True)